In [ ]:
# This is run from main working directory
import torch # for transformers

import mlrun
mlrun.set_environment(api_path="http://localhost:30070")
project = mlrun.load_project(name="legalcontractextractor", context="../")
# print(project.to_yaml())

from dotenv import load_dotenv
load_dotenv()

import os
ENV = os.environ["ENV"]

from pathlib import Path
from datetime import datetime

In [ ]:
"""
Perform the analysis for all 17 hypotheses. 

Each JSON object in the array represents the label of each hypothesis pertaining to the contract. Each object contains the id of its hypothesis, the quotes or phrases from the contract that justifies the label given (these must only come from the contract), and the label that represents its status (entailment, contradiction, not_mentioned). 

Only quote the text that justifies the label you chose for the source_clause field, even if it comes from different sections. 
"""

system_prompt = """
You are a legal contract analyst assistant that analyzes a legal contract to confirm or deny the status of a set of hypotheses about the contract. Each hypothesis is a statement about the contract and you must choose a label that represents the status of the hypothesis based on the content of the contract, and you must quote the relevant parts of the contract that support this classification. 

Read the contract and look for phrases or quotes to label each hypothesis. Label each hypothesis with:
"entailment" if the hypothesis is true
"contradiction" if the hypothesis is false
"not_mentioned" if the hypothesis cannot be confirmed or denied because there is no supporting evidence in the contract

Perform the analysis for all 17 hypotheses. The hypotheses with their corresponding ids are as follows:

nda-1: All Confidential Information shall be expressly identified by the Disclosing Party.
nda-2: Confidential Information shall only include technical information.
nda-3: Confidential Information may include verbally conveyed information.
nda-4: Receiving Party shall not use any Confidential Information for any purpose other than the purposes stated in the Agreement.
nda-5: Receiving Party may share some Confidential Information with some of Receiving Party's employees.
nda-7: Receiving Party may share some Confidential Information with some third-parties (including consultants, agents and professional advisors).
nda-8: Receiving Party shall notify Disclosing Party in case Receiving Party is required by law, regulation or judicial process to disclose any Confidential Information.
nda-10: Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.
nda-11: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
nda-12: Receiving Party may independently develop information similar to Confidential Information.
nda-13: Receiving Party may acquire information similar to Confidential Information from a third party.
nda-15: Agreement shall not grant Receiving Party any right to Confidential Information.
nda-16: Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.
nda-17: Receiving Party may create a copy of some Confidential Information in some circumstances.
nda-18: Receiving Party shall not solicit some of Disclosing Party's representatives.
nda-19: Some obligations of Agreement may survive termination of Agreement.
nda-20: Receiving Party may retain some Confidential Information even after the return or destruction of Confidential Information.

Respond with a structured JSON array of 17 JSON objects. Each JSON object in the array represents the label of each hypothesis pertaining to the contract. Each object contains the hypothesis, its id, the label that represents its status (entailment, contradiction, not_mentioned), and the source clause from the contract that justifies the label given (these must only come from the contract). 

Only quote the text that justifies the label you chose for the source_clause field, even if it comes from different sections. If the label is "not_mentioned" leave the source_clause field blank.
"""

system_prompt = system_prompt.strip()

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("JerroldK/Hermes-4-14B-contract-extractor")
print("Tokenizer loaded")
print(f"The number of tokens in the system prompt: {len(tok(system_prompt)['input_ids'])}")

vLLM inference parameters for docs
https://docs.djl.ai/master/docs/serving/serving/docs/lmi/user_guides/lmi_input_output_schema.html#additional-vllm-generation-parameters

In [ ]:
# ------- JSON OUTPUT FORMAT ----------------
# DJL nntively supports an OpenAI-compatible Chat Completions API
# This follows the vllm docs with openapi json format
structured_json_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "hypothesis_list",
        "schema": {
            "type": "object",
            "properties": {
                "Hypotheses": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "hypothesis": {"type": "string"},
                            "hypothesis_id": {"type": "string"},
                            "label": {
                                "type": "string",
                                "enum": [
                                    "entailment",
                                    "contradiction",
                                    "not_mentioned",
                                ],
                            },
                            "source_clause": {"type": "string"},
                        },
                        "required": [
                            "hypothesis",
                            "hypothesis_id",
                            "label",
                            "source_clause",
                        ],
                    },
                }
            },
            "required": ["Hypotheses"],
        },
    },
}

In [ ]:
from pydantic import BaseModel
from typing import List, Literal

# OpenAI
prompt_template = [
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": "{contract}",
    },
]

# MLRun doesn't allow storage of the chatml template used by the qwen3 models
# prompt_template = [{
#     "template": (
#         "<|im_start|>system\n"
#         f"{system_prompt}<|im_end|>\n"
#         "<|im_start|>user\n"
#         "{contract}<|im_end|>\n"
#         "<|im_start|>assistant\n"
#     )
# }]


tag = datetime.now().strftime("%Y%m%d_%H%M")  # this is the version
key = "contract_extractor_prompt"
artifact_path = f"s3://{ENV}-mlops-bucket-haviv/data/llm_prompt/{key}/{tag}"
print(f"Tag: {tag}")

"""
https://docs.vllm.ai/en/latest/features/structured_outputs/#online-serving-openai-api

https://docs.djl.ai/master/docs/serving/serving/docs/lmi/user_guides/lmi_input_output_schema.html#additional-vllm-generation-parameters

"""

invoc_config = {
    "temperature": 0.2,
    "top_p": 0.95,
    "max_new_tokens": 3000,
    "response_format": structured_json_format,
}

output = project.log_llm_prompt(
    key=key,
    prompt_template=prompt_template,
    prompt_legend={
        "issue_description": {
            "field": "contract",
            "description": "The legal contract to extract data from",
        },
    },
    invocation_config=invoc_config,
    description="Prompt template for the legal extractor",
    artifact_path=artifact_path,
    tag=tag,
)

### Validation

In [ ]:
# Retrieve the prompt
prompt = project.list_llm_prompts(name="contract_extractor_prompt", tag="20260814_1626")[0].read_prompt()
print(type(prompt[0]))

from pprint import pprint
print(prompt[0]['content'])

In [ ]:
project.get_artifact(key="contract_extractor_prompt", tag="latest").to_dict()['spec']['invocation_config']

In [ ]:
project.get_artifact(key="contract_extractor_prompt", tag="latest").to_dict()['spec']['invocation_config']['temperature']

In [ ]:
project.get_artifact(key="contract_extractor_prompt", tag="latest").to_dict()['spec']['target_path']

In [ ]:
import json

print(
    json.dumps(
        project.get_artifact(key="contract_extractor_prompt", tag="latest").to_dict()['spec']['invocation_config']
    ).replace("\'", '\'')
)


In [ ]:
# get the id of a specific version
# version is specified with the tag which is "latest" or "yyyymmdd"
project.get_artifact(key="contract_extractor_prompt", tag="latest").to_dict()['spec']['producer']['tag']

### Delete LLM PROMPT (UI Doesn't allow this)

In [ ]:
# The only way to properly delete a data artifact and its historical versions
import os

artifact = project.get_artifact(key="contract_extractor_prompt")
project.delete_artifact(
    artifact,
    deletion_strategy='metadata-only',
    secrets={
        "AWS_ACCESS_KEY_ID": os.environ["AWS_ACCESS_KEY_ID"],
        "AWS_SECRET_ACCESS_KEY": os.environ["AWS_SECRET_ACCESS_KEY"],
    },
)